<a href="https://colab.research.google.com/github/amirgroup-codes/ProtoMech/blob/main/ProtoMech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="left">
  <img src="https://raw.githubusercontent.com/amirgroup-codes/ProtoMech/main/ProtoMech_Logo_Glow.svg"
       alt="ProtoMech"
       width="60%">
</p>

# ProtoMech: Protein Circuit Tracing via Cross-layer Transcoders</h1>

ProtoMech is a framework for discovering computational circuits in protein language models using cross-layer transcoders. This colab notebook is designed to produce the four files required for our [website](https://protmech.github.io/):
1. `activation_indices.json`
2. `seq.txt`
3. `top_activations.json`
4. `virtual_weights.json`

A link to the paper can be found [here](https://arxiv.org/abs/2602.12026). We additionally provide our [code](https://github.com/amirgroup-codes/ProtoMech), [models](https://huggingface.co/ktalreja/ProtoMechModels), and [data](https://huggingface.co/datasets/ktalreja/ProtoMechData).

---

In [ ]:
# @title 0. Check GPU status and install dependencies
import torch
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Code may run extremely slowly.")
    print("----------------------------------------------------------------")
    print("TO FIX THIS:")
    print("1. Click 'Runtime' in the top menu.")
    print("2. Select 'Change runtime type'.")
    print("3. Under 'Hardware accelerator', select 'T4 GPU'.")
    print("4. Click 'Save' and then re-run this cell.")
    print("----------------------------------------------------------------")

import os
import sys
import shutil
import tarfile
import subprocess
from google.colab import files
from huggingface_hub import hf_hub_download
from IPython.display import clear_output
import pty
!pip install -q huggingface_hub torch pytorch-lightning fair-esm

### 1.5 Discover your own circuit (optional)
Use this section to train a probe and discover a circuit for your own custom dataset. We follow a similar protocol to Appendix D of the paper.

### **Input CSV format**
Your CSV must have two specific columns (not case-sensitive):
1.  **Sequence**: `sequence` or `mutated_sequence`.
2.  **Score**: `score`, `DMS_score`, or `class`

#### **Option A: Binary classification**
* **Sequence:** can vary in length.
* **Score:** Must contain **only** `0` and `1`.
* **Example:**
    ```csv
    sequence,class
    MKV...AAA,1
    AGL...TTV,0
    MKV...AAB,1
    ```

#### **Option B: Regression**
* **Sequence:** must be same length.
* **Score:** Continuous numbers (float).
* **Example:**
    ```csv
    mutant,mutated_sequence,DMS_score
    K3R,MSR...LYK,3.74
    K3Q,MSQ...LYK,3.75
    K3E,MSE...LYK,3.67
    ```

In [ ]:
# @title Run circuit discovery
# @markdown **Settings**
import os
import sys
import subprocess
import pty
import torch
import argparse
from google.colab import files
if hasattr(torch.serialization, 'add_safe_globals'):
    torch.serialization.add_safe_globals([argparse.Namespace])

# @markdown Select the type of task:
task_type = "Binary classification" # @param ["Binary classification", "Regression"]
# @markdown Select the ESM2 model:
model_choice = "ESM2-35M" # @param ["ESM2-8M","ESM2-35M"]
# @markdown Check to upload your CSV file:
upload_csv = True # @param {type:"boolean"}
# @markdown Output folder name:
output_dir = "custom_circuit" # @param {type:"string"}
# @markdown Where to save the results:
external_path = "/content/experiments" # @param {type:"string"}

REPO_ROOT = "/content/ProtoMech"
SCRIPT_PATH = os.path.join(REPO_ROOT, "visualization", "auto_discover_circuit_website.py")

if model_choice == "ESM2-8M":
    esm_weights = os.path.join(REPO_ROOT, "models", "esm2_t6_8M_UR50D.pt")
    clt_checkpoint = os.path.join(REPO_ROOT, "models", "CLT_L6_D3200", "checkpoints", "last.ckpt")
    family_dir = os.path.join(REPO_ROOT, "family_circuit", "families")
    function_dir = os.path.join(REPO_ROOT, "function_circuit", "functions")
    activations_pt = os.path.join(REPO_ROOT, "visualization", "top10_activations.pt")
else:
    esm_weights = os.path.join(REPO_ROOT, "models", "esm2_t12_35M_UR50D.pt")
    clt_checkpoint = os.path.join(REPO_ROOT, "models", "CLT_L12_D4800", "checkpoints", "last.ckpt")
    family_dir = os.path.join(REPO_ROOT, "family_circuit", "families_35M")
    function_dir = os.path.join(REPO_ROOT, "function_circuit", "functions_35M")
    activations_pt = os.path.join(REPO_ROOT, "visualization", "top10_activations_35M.pt")
print(f"Using model {model_choice} -> ESM weights: {esm_weights}, CLT checkpoint: {clt_checkpoint}")

# --- 1. Input Handling ---
if upload_csv:
    print("Please upload your CSV file...")
    uploaded = files.upload()
    if not uploaded:
        sys.exit("Upload cancelled.")
    csv_name = list(uploaded.keys())[0]
    csv_path = os.path.join(os.getcwd(), csv_name)
else:
    sys.exit("Check 'upload_csv' to proceed.")
is_binary = (task_type == "Binary classification")
entry_name = os.path.splitext(csv_name)[0]
full_output_dir = os.path.join(external_path, output_dir)

# --- 2. Run Command ---
cmd = [
    "python", SCRIPT_PATH,
    "--csv_path", csv_path,
    "--is_binary", str(is_binary),
    "--output_dir", full_output_dir,
    "--entry_name", entry_name,
    "--clt_checkpoint", clt_checkpoint,
    "--esm_weights", esm_weights,
    "--model_size", model_choice,
    "--batch_size", "8"
]
print(f"\nStarting Discovery ({task_type})...")
print(f"   Input: {csv_name}")
print(f"   Output: {full_output_dir}/{entry_name}.json\n")
master, slave = pty.openpty()
p = subprocess.Popen(cmd, stdout=slave, stderr=slave, close_fds=True)
os.close(slave)
try:
    while True:
        try:
            data = os.read(master, 1024).decode()
            if not data: break
            sys.stdout.write(data)
            sys.stdout.flush()
        except OSError: break
except Exception: pass
p.wait()
os.close(master)

if p.returncode == 0:
    print(f"\nDone! Download your JSON here: {full_output_dir}/{entry_name}.json")
else:
    print("\nDiscovery Failed.")

In [ ]:
# @title 2. Add sequences to compute
# @markdown Example sequences:
# @markdown - Seq 1 (wildtype): `QYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE`
# @markdown - Seq 2: `QYKLILNGKTLKGETTTEAVDAWTAEKVFKQYANDNGVDGEWTYDDATKTFTVTE`
# @markdown - Seq 3: `QYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEQTYDDATKTFTVTE`

# @markdown Note: Use `Add Sequence +` to compare variations of the same protein (e.g., assessing the effect of mutations on a wildtype sequence). To discover a completely new circuit for a different protein, please reset and run the discovery pipeline again.
import ipywidgets as widgets
from IPython.display import display

class SequenceInputManager:
    def __init__(self):
        self.sequences = []
        self.container = widgets.VBox()
        self.add_button = widgets.Button(description="Add Sequence +", icon="plus")
        self.add_button.on_click(self.add_field)

        # Initial field
        self.add_field(None)

    def add_field(self, b):
        idx = len(self.sequences) + 1
        label = "Seq 1 (wildtype):" if idx == 1 else f"Seq {idx}:"
        text = widgets.Text(placeholder=f"Enter protein sequence {idx}...", layout=widgets.Layout(width='80%'))
        box = widgets.HBox([widgets.Label(value=label, layout=widgets.Layout(width='120px')), text])
        self.sequences.append(text)
        self.container.children = tuple(list(self.container.children) + [box])

    def get_sequences(self):
        return [w.value.strip() for w in self.sequences if w.value.strip()]

    def display(self):
        display(widgets.VBox([self.container, self.add_button]))

# Instantiate and display
seq_manager = SequenceInputManager()
seq_manager.display()

In [ ]:
# @title 3. Generate Circuit Files (Run after adding sequences)
# @markdown **Configuration**
# @markdown - `circuit` (optional): Leave empty to auto-generate from `Seq 1`. You can also find a list of pre-discovered circuits [here](https://github.com/amirgroup-codes/ProtoMech/blob/main/visualization/circuits.md).
# @markdown - `upload_custom_circuit`: Check to upload a custom JSON.
# @markdown - `output_dir`: Name of the folder to create.

import os
import sys
import subprocess
import shutil
from google.colab import files

# --- 1. Get Inputs from Widget ---
sequences = seq_manager.get_sequences()
circuit = "SPG1_STRSG_Olson_2014" # @param {type:"string"}
upload_custom_circuit = False # @param {type:"boolean"}
output_dir = "GB1" # @param {type:"string"}
external_path = "/content/experiments" # @param {type:"string"}

# --- Validation ---
if not sequences:
    print("❌ Error: No sequences provided. Please add sequences in the UI above.")
    sys.exit(1)
print(f"Processing {len(sequences)} sequences...")
print(f"   Seq 1 (wldtype): {sequences[0][:10]}...")

# --- Setup Paths ---
REPO_ROOT = "/content/ProtoMech"
SCRIPTS_DIR = os.path.join(REPO_ROOT, "visualization")
FULL_OUTPUT_DIR = os.path.join(external_path, output_dir)
os.makedirs(FULL_OUTPUT_DIR, exist_ok=True)
probe_path = os.path.join(FULL_OUTPUT_DIR, f"{entry_name}_probe.pt")

if not os.path.exists(SCRIPTS_DIR):
    raise FileNotFoundError(f"Could not find directory: {SCRIPTS_DIR}")
os.chdir(SCRIPTS_DIR)
def run_realtime(command):
    """
    Runs a command with a pseudo-terminal to allow real-time
    output and progress bars in Colab.
    """
    master, slave = pty.openpty()
    p = subprocess.Popen(command, stdout=slave, stderr=slave, close_fds=True)
    os.close(slave)
    try:
        while True:
            try:
                data = os.read(master, 1024)
                if not data: break
            except OSError:
                break
    except Exception as e:
        pass #
    p.wait()
    os.close(master)
    return p.returncode, ""



circuit_json_path = None
needs_generation = False

# CASE A: User wants to upload a file
if upload_custom_circuit:
    print("\nPlease upload custom circuit json...")
    uploaded = files.upload()
    if not uploaded:
        print("Upload cancelled. Aborting.")
        sys.exit(1)
    filename = list(uploaded.keys())[0]
    target_path = os.path.join(FULL_OUTPUT_DIR, filename)
    os.rename(filename, target_path)
    circuit_json_path = target_path

# CASE B: User specified a circuit query
elif circuit.strip():
    clean_query = circuit.strip()
    json_filename = clean_query if clean_query.endswith(".json") else f"{clean_query}.json"
    base_name = os.path.splitext(clean_query)[0]

    if clean_query.startswith("IPR"):
        candidate_path = os.path.join(family_dir, "CLT_sequential", json_filename)
        if os.path.exists(candidate_path):
            circuit_json_path = candidate_path
        else:
            print(f"Warning: Could not find family circuit at {candidate_path}. Auto-generating...")
            needs_generation = True
    else:
        candidate_path = os.path.join(function_dir, "CLT_sequential", "multiples", base_name, "rand_multiples_fold0.json")
        if os.path.exists(candidate_path):
            circuit_json_path = candidate_path
        else:
            print(f"Warning: Could not find function file at {candidate_path}. Auto-generating...")
            needs_generation = True

# CASE C: No input provided
else:
    print("Auto-generating circuit from seq 1...")
    needs_generation = True



# --- Pipeline Execution ---
# Step 0: Generate Circuit (from Seq 1)
if needs_generation:
    print("\n[Step 0] Generating circuit JSON...")
    generated_json_path = os.path.join(FULL_OUTPUT_DIR, f"{output_dir}_circuit.json")
    cmd = [
        "python", "circuit_top_acts.py",
        "--sequence", sequences[0],
        "--output", generated_json_path
    ]
    exit_code, cmd_output = run_realtime(cmd)
    if exit_code != 0:
        print("Error generating circuit")
        circuit_json_path = None
    else:
        circuit_json_path = generated_json_path

# Step 1: Analyze All Sequences
if circuit_json_path:
    print("\n[Step 1] Running multi-sequence analysis...")
    if os.path.exists(output_dir):
        if os.path.islink(output_dir):
            os.unlink(output_dir)
        elif os.path.isdir(output_dir):
            shutil.rmtree(output_dir)
    os.symlink(FULL_OUTPUT_DIR, output_dir)
    cmd_analysis = [
        "python", "circuit_analysis_builder_website.py",
        "--entry_name", output_dir,
        "--circuit_json", circuit_json_path,
        "--clt_ckpt", clt_checkpoint,
        "--esm_path", esm_weights,
        "--activations_pt", activations_pt,
        "--sequences"
    ] + sequences

    if os.path.exists(probe_path):
        cmd_analysis[4:4] = ["--probe_path", probe_path]
    else:
        print(f"No probe file found at {probe_path}; running analysis without probe.")
    exit_code, cmd_output = run_realtime(cmd_analysis)

    if exit_code != 0:
        print("Error conducting circuit analysis")
    else:
        print("\n[Step 2] Computing edge weights for each sequence (may take some time)...")

        subfolders = [f.path for f in os.scandir(FULL_OUTPUT_DIR) if f.is_dir() and "seq" in f.name]
        subfolders.sort() # Ensure seq1, seq2 order
        for folder_path in subfolders:
            folder_name = os.path.basename(folder_path)
            print(f"\n   Processing {folder_name}...")
            target_rel_path = os.path.join(output_dir, folder_name)
            cmd_weights = [
                "python", "get_edge_weights.py",
                "--base_folder", target_rel_path,
                "--clt_ckpt", clt_checkpoint,
                "--esm_path", esm_weights
            ]
            w_exit, w_out = run_realtime(cmd_weights)
            if w_exit != 0:
                print(f"   ❌ Failed to compute weights for {folder_name}")

        print("\n==========")
        print(f"Results saved to: {FULL_OUTPUT_DIR}")
        print("==========")
        print("Files generated:")
        for f in os.listdir(FULL_OUTPUT_DIR):
             print(f" - {f}")

    # Cleanup Symlink
    if os.path.islink(output_dir):
        os.unlink(output_dir)
else:
    if not needs_generation:
        print("Aborted: No valid circuit JSON found.")